In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(67)

In [305]:
class NN(torch.nn.Module):
    def __init__(self, input_dim, output_dim, layers, neurons, activation = 'tanh'):
        super(NN, self).__init__()
        self.activation = activation

        self.input_layer   = torch.nn.Linear(input_dim, neurons, bias = True)
        self.hidden_layers = torch.nn.ModuleList([torch.nn.Linear(neurons, neurons, bias = True) for _ in range(layers)])
        self.output_layer  = torch.nn.Linear(neurons, output_dim, bias = True)
        
        self.act           = torch.nn.Tanh()
    
    def forward(self, x):
        a =  self.input_layer(x)
        for l in self.hidden_layers:
            a = self.act(l(a))
        return self.output_layer(a)
    
    
    def init_xavier(self):
        def init_weights(m):
            if type(m) == nn.Linear and m.weight.requires_grad and m.bias.requires_grad:
                g = nn.init.calculate_gain(self.activation)
                torch.nn.init.xavier_uniform_(m.weight, gain=g)
                m.bias.data.fill_(0)
        self.apply(init_weights)

sum(p.numel() for p in NN(5, 5, 5, 5).parameters())

210

In [ ]:
BATCH_SIZE = 32

In [307]:
def u_0(x):
    torch.sin(x)

def v_0(x):
    torch.sin(x)
    

domain = torch.tensor([[0, 1], [0, 2]]) # x = [0, 1]; t = [0, 2]
sobol_engine = torch.quasirandom.SobolEngine(dimension = 2) # x and t

class dataset(torch.utils.data.Dataset):
    def __init__(self, engine: torch.quasirandom.SobolEngine, draw: int):
        self.data = engine.draw(draw) 

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index]
    
dataset_obj = dataset(sobol_engine, 10)
dataloader = torch.utils.data.DataLoader(dataset_obj, batch_size = 32, shuffle = True)
next(iter(dataloader))

tensor([[0.2500, 0.7500],
        [0.1875, 0.3125],
        [0.8750, 0.8750],
        [0.1250, 0.6250],
        [0.6875, 0.8125],
        [0.3750, 0.3750],
        [0.6250, 0.1250],
        [0.0000, 0.0000],
        [0.5000, 0.5000],
        [0.7500, 0.2500]])

In [331]:
domain = torch.tensor([[0, 1], [0, 2]]) # x = [0, 1]; t = [0, 2]
sobol_engine = torch.quasirandom.SobolEngine(dimension = 2) # x and t

def u_0(x):
    return torch.sin(x)

def v_0(x):
    return torch.cos(x)
    
def get_interior(engine, x):
    drew = engine.draw(x)
    drew[:, 0] =  domain[0][0] + (domain[0][1] - domain[0][0]) * drew[:, 0]
    drew[:, 1] =  domain[1][0] + (domain[1][1] - domain[1][0]) * drew[:, 1]
    return drew

def get_BC(engine, x):
    data  = engine.draw(x)
    data[:,1] = domain[1][0] + (domain[1][1] - domain[1][0]) * data[:,1]

    # x = 0
    data_0 = data.clone()
    data_0[:,0] = domain[0][0]

    # x = L
    data_L = data.clone()
    data_L[:,0] = domain[0][1]

    return data_0, data_L

def get_initial(engine, x):
    data  = engine.draw(x)
    data[:,0] = domain[0][0] + (domain[0][1] - domain[0][0]) * data[:,0]

    data[:, 1] = 0 # t = 0 

    x = data[:,0]
    u = u_0(x)
    u_t = v_0(x)

    u = torch.stack((u, torch.zeros_like(u)), dim=1)
    u_t = torch.stack((u_t, torch.zeros_like(u_t)), dim=1)

    return u, u_t

get_BC(sobol_engine, 10)

(tensor([[0.0000, 0.0000],
         [0.0000, 1.0000],
         [0.0000, 0.5000],
         [0.0000, 1.5000],
         [0.0000, 0.7500],
         [0.0000, 1.7500],
         [0.0000, 0.2500],
         [0.0000, 1.2500],
         [0.0000, 0.6250],
         [0.0000, 1.6250]]),
 tensor([[1.0000, 0.0000],
         [1.0000, 1.0000],
         [1.0000, 0.5000],
         [1.0000, 1.5000],
         [1.0000, 0.7500],
         [1.0000, 1.7500],
         [1.0000, 0.2500],
         [1.0000, 1.2500],
         [1.0000, 0.6250],
         [1.0000, 1.6250]]))

In [309]:
u_predictor_model = NN(input_dim = 2, output_dim = 1, layers = 4, neurons = 20)
E_predictor_model = NN(input_dim = 1, output_dim = 1, layers = 4, neurons = 20)

In [397]:
# interior_data = get_interior(sobol_engine, 10)

def get_interior_residual(u_model, E_model, points):
    points.requires_grad = True

    u = u_model(points)
    grad_u = torch.autograd.grad(u.sum(), points, create_graph = True)[0]
    u_x = grad_u[:, 0]
    u_t = grad_u[:, 1]

    u_tt = torch.autograd.grad(u_t.sum(), points, create_graph=True)[0][:, 1]

    E = E_model(u_x.unsqueeze(1))
    E = E.squeeze() * u_x

    interior_residual = u_tt - torch.autograd.grad((E * u_t.unsqueeze(1)).sum(), points, create_graph=True)[0][:, 0]
    return interior_residual

interior_data = get_interior(sobol_engine, 10) 
get_interior_residual(u_predictor_model, E_predictor_model, interior_data)

tensor([ 0.0025,  0.0040,  0.0015,  0.0040, -0.0004,  0.0029,  0.0020,  0.0040,
         0.0002,  0.0033], grad_fn=<SubBackward0>)

In [398]:
def get_IC_residue(u_model, u_0, v_0, points):

    u_0_data = points[0].clone().detach().requires_grad_(True) # u at t = 0
    u_t_data = points[1].clone().detach().requires_grad_(True) # u at t = 0

    u = u_model(u_0_data) 
    ic_1_residue = u - u_0(u_0_data[:, 0].unsqueeze(1))

    u_t = torch.autograd.grad(u.sum(), u_0_data, create_graph=True)[0][:, 1]
    ic_2_residue = u_t.unsqueeze(1) - v_0(u_t_data[:, 0].unsqueeze(1))

    return ic_1_residue, ic_2_residue

IC_data = get_initial(sobol_engine, 10) # u, u_t (exact at t = 0)

get_IC_residue(u_predictor_model, u_0, v_0, IC_data)

(tensor([[-0.4028],
         [-0.7424],
         [-0.5992],
         [-0.1654],
         [-0.2274],
         [-0.6401],
         [-0.7698],
         [-0.4565],
         [-0.3465],
         [-0.7117]], grad_fn=<SubBackward0>),
 tensor([[-0.6189],
         [-0.8243],
         [-0.7114],
         [-0.5666],
         [-0.5752],
         [-0.7387],
         [-0.8527],
         [-0.6389],
         [-0.6015],
         [-0.7956]], grad_fn=<SubBackward0>))

In [399]:
def get_BC_residue(u_model, points):
    return u_model(BC_data[0]),  u_model(BC_data[1])

BC_data = get_BC(sobol_engine, 10) # t at u = 0 and t at u = L

get_BC_residue(u_predictor_model, BC_data)

(tensor([[-0.0337],
         [-0.0521],
         [-0.0326],
         [-0.0510],
         [-0.0420],
         [-0.0593],
         [-0.0466],
         [-0.0631],
         [-0.0373],
         [-0.0553]], grad_fn=<AddmmBackward0>),
 tensor([[-0.0514],
         [-0.0695],
         [-0.0503],
         [-0.0685],
         [-0.0596],
         [-0.0765],
         [-0.0641],
         [-0.0801],
         [-0.0549],
         [-0.0726]], grad_fn=<AddmmBackward0>))

In [400]:
optimizer = torch.optim.Adam(
    list(u_predictor_model.parameters()) + 
    list(E_predictor_model.parameters()),
    lr=1e-3
)

for epoch in range(1000):

    optimizer.zero_grad()
    interior_data = get_interior(sobol_engine, 10) 
    IC_data       = get_initial(sobol_engine, 10) # u, u_t (exact at t = 0)
    BC_data       = get_BC(sobol_engine, 10) # t at u = 0 and t at u = L


    interior_residue  = get_interior_residual(u_predictor_model, E_predictor_model, interior_data)
    IC_residue        = get_IC_residue(u_predictor_model, u_0, v_0, IC_data)
    BC_residue        = get_BC_residue(u_predictor_model, BC_data)


    loss_pde = torch.mean(interior_residue**2)
    loss_bc  = torch.mean((IC_residue[0]**2) + (IC_residue[1]**2))
    loss_ic  = torch.mean((BC_residue[0]**2) + (BC_residue[1]**2))

    # total
    loss = loss_pde + loss_ic + loss_bc
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(epoch, loss.item())

0 0.83489590883255
100 0.23474116623401642
200 0.23996272683143616
300 0.17778106033802032
400 0.16754236817359924
500 0.15974678099155426
600 0.15677322447299957
700 0.134614959359169
800 0.14387401938438416
900 0.14914792776107788


In [384]:
loss_ic

tensor(0.0070, grad_fn=<MeanBackward0>)